# Lesson 26 - Pydantic and FastAPI

**Goals:** Validate API boundaries, serve a prediction, inject dependencies, and test requests without a server.

**Before you begin:** Lessons 8, 11, 13, 16, and 17 cover validation, modules, composition, hints, and tests. Previous stage: [25_pytorch_deep_learning.ipynb](25_pytorch_deep_learning.ipynb).
This lesson defines all its own inputs and runs independently on Python 3.11+.

**Setup:** `requirements.txt` (Pydantic v2, FastAPI, HTTPX; Uvicorn for optional serving). No PyTorch needed here. See the README shared requirements instructions. Install from a terminal only when you choose;
these notebooks never install packages. No core example needs a network, API key, database server, or download.
All records are fictional.

Run cells top to bottom with **Shift+Enter**. Predict the output before running each example,
change one input, rerun the relevant cells, and explain what changed. Restart the kernel and
Run All after restoring the original inputs. Practice cells intentionally contain no answers.
Attempt the exercises and project before opening `instructor_solutions/26_pydantic_and_fastapi_solutions.ipynb`.


## 1. Runtime contracts with Pydantic v2

Type hints alone do not validate runtime input. Pydantic constructs typed models, parses
compatible input, checks constraints, and reports field errors. `model_validate` parses Python
data; `model_validate_json` parses JSON. `model_dump` returns Python data; `model_dump_json`
serializes it. We reject extra fields so spelling mistakes do not disappear.
Parsing is intentionally flexible here: numeric text becomes a float. Use strict fields/config
when coercion is inappropriate. Predict which fields use defaults.


In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

class PredictionRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)
    hours: float = Field(ge=0, le=24)
    temperature: float = Field(default=25, ge=-20, le=60)
    site: str = Field(default="demo", min_length=1, max_length=40)

    @field_validator("site")
    @classmethod
    def nonblank_site(cls, value):
        if not value.strip():
            raise ValueError("Site must not be blank")
        return value.strip()

request = PredictionRequest.model_validate({"hours":"4"})
print(request.model_dump(), request.model_dump_json())
assert request.hours == 4. and request.temperature == 25
assert PredictionRequest.model_validate_json(request.model_dump_json()) == request
for invalid in [{"hours":-1}, {"hours":float("nan")}, {"hours":2,"typo":1}, {"hours":1,"site":" "}]:
    try:
        PredictionRequest.model_validate(invalid)
    except ValidationError as error:
        print(error.errors()[0]["loc"], error.errors()[0]["type"])


## 2. A replaceable prediction service

An API boundary should not contain training logic. Put inference behind a service object
and compose it into the application. This offline service uses a known formula so API tests
have exact expected values; it is not a trained model and does not claim learned accuracy.
To serve Lesson 25's network, replace this service, loading trusted weights once at startup
and applying its exact feature normalization. Keep training out of request handlers.

The formula is clamped at zero so all validated temperature/hour combinations produce nonnegative energy.


In [ ]:
class PredictionResponse(BaseModel):
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)
    site: str
    kwh: float = Field(ge=0)
    model_version: str

class EnergyService:
    def predict(self, request):
        return PredictionResponse(site=request.site,
            kwh=max(0., 5 + 3*request.hours + 0.5*request.temperature), model_version="formula-v1")

service = EnergyService()
assert service.predict(PredictionRequest(hours=4)).kwh == 29.5
# Valid cold conditions must still satisfy the nonnegative output contract.
assert service.predict(PredictionRequest(hours=0,temperature=-20)).kwh == 0


## 3. Routes, parameters, responses, and decorators

HTTP requests name a method and path. A path parameter identifies a resource; a query parameter
controls an option; a body carries structured input. The `@app.get`/`@app.post` syntax is a
decorator: it registers the function with FastAPI when the definition runs. You need not implement
decorators to use this registration pattern. A response model documents and validates returned data.
200 means a successful read/processing operation; 201 means creation; 404 means a resource is absent;
422 means request validation failed. Predict the JSON for the demo site's query.


In [ ]:
from typing import Annotated
from fastapi import FastAPI, Depends, HTTPException, Query
from fastapi.testclient import TestClient

app = FastAPI(title="Fictional energy API")
sites = {101:"demo"}

@app.get("/sites/{site_id}")
def read_site(site_id: int, uppercase: bool = False):
    if site_id not in sites:
        raise HTTPException(status_code=404, detail="Site not found")
    name = sites[site_id]
    return {"id":site_id,"name":name.upper() if uppercase else name}

@app.get("/limits")
def read_limits(max_hours: Annotated[int, Query(ge=1, le=24)] = 8):
    return {"max_hours":max_hours}

def get_service():
    return service

@app.post("/predict", response_model=PredictionResponse, status_code=200)
def predict(body: PredictionRequest, predictor: Annotated[EnergyService, Depends(get_service)]):
    return predictor.predict(body)


## 4. Test endpoints without opening a network port

TestClient runs requests against the application in-process. The context manager handles
application startup/shutdown. Check status and body, not only that a function can be called.
Framework validation of an invalid body produces 422 before your handler runs.
Change the request temperature and predict the new result before rerunning.


In [ ]:
with TestClient(app) as client:
    response = client.post("/predict", json={"hours":4, "site":"north"})
    assert response.status_code == 200
    assert response.json() == {"site":"north","kwh":29.5,"model_version":"formula-v1"}
    assert client.get("/sites/101?uppercase=true").json()["name"] == "DEMO"
    assert client.get("/sites/999").status_code == 404
    assert client.get("/sites/not-an-int").status_code == 422
    assert client.get("/limits?max_hours=25").status_code == 422
    assert client.post("/predict", json={"hours":-1}).status_code == 422
print("Normal, missing-resource, and invalid-input API checks passed.")


## 5. Dependency injection makes service swaps testable

Depends tells FastAPI to obtain a dependency and supply it to the handler. This is the
composition pattern from Lesson 13 applied at a request boundary. Tests override the provider
with a deterministic fake, then remove the override in finally. Avoid leaking test state into
later requests. HTTPException is for deliberate client-facing HTTP errors; unrelated programming
failures should remain visible in tests.


In [ ]:
class FixedService:
    def predict(self, request):
        return PredictionResponse(site=request.site,kwh=7,model_version="test")

app.dependency_overrides[get_service] = lambda: FixedService()
try:
    with TestClient(app) as client:
        response = client.post("/predict",json={"hours":4})
        assert response.json()["kwh"] == 7
        assert response.json()["model_version"] == "test"
finally:
    app.dependency_overrides.pop(get_service, None)
assert not app.dependency_overrides


## 6. Async is cooperative waiting

Calling an async function creates a coroutine; `await` runs it until it yields or completes.
While an async I/O operation waits, an event loop can run other tasks. `asyncio.sleep` below
simulates such waiting without a network. It does not make numerical computation faster.
Use ordinary `def` for this tiny synchronous prediction handler. Use async handlers with libraries
that expose awaitable I/O. A blocking network call or heavy training inside async def blocks the loop.
FastAPI runs ordinary handlers in a thread pool; large CPU work needs a separate worker strategy.

Jupyter already runs an event loop, so use top-level await there. In an ordinary Python script,
use `asyncio.run(...)` under the main guard instead.


In [ ]:
import asyncio

async def fetch_status(label):
    await asyncio.sleep(0)  # Yield control; no real I/O in this demonstration.
    return {"source":label,"status":"ready"}

async def collect_status():
    return await asyncio.gather(fetch_status("sensor"),fetch_status("model"))

statuses = await collect_status()
assert [item["source"] for item in statuses] == ["sensor","model"]
print(statuses)


## 7. Project structure and configuration

Move definitions into modules using Lesson 11's import boundaries:

```text
energy_api/
    __init__.py
    schemas.py       # Pydantic request/response contracts
    service.py       # Formula or trusted model inference; no HTTP dependencies
    config.py        # Read and validate environment settings once
    main.py          # app, routes, and dependency providers
tests/
    test_api.py      # TestClient requests and dependency overrides
```

Read `MODEL_VERSION` or artifact paths with `os.environ.get` in config, then inject configuration
into the service. Keep secrets out of notebooks, source, responses, and logs. Load a model once
per application process rather than per request. Imports must not launch a server or train a model.

After extracting `app` to `energy_api/main.py`, optionally run
`python -m uvicorn energy_api.main:app --reload` in a terminal and visit the local `/docs` page.
No server is needed for these tests. Uvicorn cannot import a notebook as a normal module.

## 8. Common mistakes and debugging

Use Pydantic v2: `model_dump` replaces older `.dict()` examples. Inspect 422 error locations to
distinguish path, query, and body issues. Don't return an error dictionary with status 200.
Avoid calling handlers directly to test dependency injection: send requests with TestClient.
Do not run asyncio.run inside the notebook's event loop. Check the selected kernel environment
when imports fail; installing into another interpreter will not fix the notebook kernel.


## Practice lab

Use the supplied inputs, add your own comments, and compare with the check values. Use `practice_`
names where helpful. Increase the difficulty one exercise at a time. Checks describe results;
they are not complete implementations. Test a boundary or malformed input when applicable.


### Exercise 1. Validate defaults and bad input

Parse `{"hours":"2","site":" west "}`; test hours 25 and an unknown field.

**Check:** hours 2.0, site west, temperature 25; both invalid cases raise ValidationError.


In [ ]:
practice_data = {"hours":"2","site":" west "}
# TODO: parse, serialize, and test invalid cases.


### Exercise 2. Test path and query parameters

Request site 101 with uppercase true, site 999, and limits max_hours=0.

**Check:** DEMO/200, 404, 422 respectively.


In [ ]:
# TODO: use a TestClient context and assert status and meaningful body fields.


### Exercise 3. Predict and serialize

POST hours 2, temperature 20, site west. Validate the response with PredictionResponse.

**Check:** kwh 21, model_version formula-v1; round-trip serialization preserves fields.


In [ ]:
practice_body = {"hours":2,"temperature":20,"site":"west"}
# TODO: POST, status, response validation, JSON round trip.


### Exercise 4. Inject a service double

Override get_service with one returning 99 kWh and version exercise. Ensure the override is removed and normal inference resumes.

**Check:** 99 under override; 29.5 after cleanup for hours 4 at default temperature.


In [ ]:
class PracticeService:
    def predict(self, request):
        # TODO: return the response contract.
        pass
# TODO: override, test, finally cleanup, retest.


### Exercise 5. Compose asynchronous waits

Write an async function that awaits fetch_status for archive and index concurrently with gather.

**Check:** two results in input order, both ready. Explain why adding CPU-heavy training would not benefit.


In [ ]:
async def practice_statuses():
    # TODO: gather two awaitable operations.
    pass
# TODO: top-level await and check.


## Chapter project

Create a separate FastAPI app with POST `/batch-predict`, accepting an object with a list of
1–5 PredictionRequest objects. Return a list of PredictionResponse objects using an injected
service. Keep this stateless: 200 is appropriate because nothing is stored.

**Check:** hours 0 and 4 at temperature 25 return 17.5 and 29.5; empty/six-item batches or a negative
hour return 422. Override the provider once to test isolation. Describe the modules you would extract.


In [ ]:
# TODO: BatchRequest, separate app, injected handler, response model, endpoint tests.


## Reflection and knowledge check

1. How do type hints and runtime validation differ?
2. Which failures produce 404 versus 422?
3. Why inject an inference service?
4. When should a handler use async def?

Choose one changed input, predict the new result, and explain your observation to a colleague.


## Official references

- [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/)
- [FastAPI testing](https://fastapi.tiangolo.com/tutorial/testing/)
- [Dependencies](https://fastapi.tiangolo.com/tutorial/dependencies/)
- [Async guidance](https://fastapi.tiangolo.com/async/)
